In [ ]:
import os
import numpy as np
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize
from langchain.llms import OpenAI

In [ ]:
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
import openai
from langchain.document_loaders import TextLoader
from langchain.indexes import VectorstoreIndexCreator

In [ ]:
os.environ['OPEAI_API_KEY']=''

In [ ]:
openai.api_key=''

In [ ]:
directory=''

In [ ]:
def read_files(directory, chunk_size):
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            with open(os.path.join(directory, filename), 'r') as file:
                data = file.read()
                chunks = [data[i:i+chunk_size] for i in range(0, len(data), chunk_size)]
                yield filename, chunks

In [ ]:
def retrieve_documents(query, directory):
    query_tokens = word_tokenize(query.lower())
    scores = bm25_obj.get_scores(query_tokens)
#     top_doc_idx = np.argmax(scores)
    top_docs_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:3]
    cntxt=[]
    for indx in top_docs_indices:
        cntxt.append(TextLoader(os.path.join(directory, documents[indx][0])).load()[0])
    return cntxt

In [ ]:
documents = list(read_files(directory, chunk_size=1000))

In [ ]:
tokenized_docs = [word_tokenize(chunk.lower()) for _, chunks in documents for chunk in chunks]

In [ ]:
bm25_obj = BM25Okapi(tokenized_docs)

In [ ]:
prompt='''Give the appropriate answer for the given question using the context from the document.Be interactive while giving the answer.Greet before starting to give answer
Give a complete answer for the question and be grammatically correct.The answer should be specific to India if nothing is written about the country in the question.If you don't know the answer,Just say "Sorry,I don't know".

Context:{}

Question:{}
'''

In [ ]:
ground_truth=pd.read_csv('')
ground_truth=ground_truth.drop(columns=['Unnamed: 0'])
ground_truth.head(3)

In [ ]:
from langchain.chains.question_answering import load_qa_chain

In [ ]:
llm=OpenAI(temperature=0,openai_api_key=os.environ['OPENAI_API_KEY'])

In [ ]:
chain = load_qa_chain(llm, chain_type="stuff")

In [ ]:
rows=[]
for i in tqdm(range(len(ground_truth))):
    query=ground_truth['question'][i].strip()
    cntxt=retrieve_documents(query,directory)
    input_query=prompt.format(cntxt,query)
    result=chain.run(input_documents=cntxt, question=prompt+query).strip()
    rows.append([ground_truth['title'][i],ground_truth['question'][i],ground_truth['ground_truth'][i],result.strip(),0])
    time.sleep(3)

In [ ]:
import csv
fields=['title','question','ground_truth','answer_generated','score']
with open('', 'w+') as f:
    # using csv.writer method from CSV package
    write = csv.writer(f)
    write.writerow(fields)
    write.writerows(rows)
f.close()